In [ ]:
!pip install transformers peft accelerate bitsandbytes>=0.46.1

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import transformers
transformers.logging.set_verbosity_error()

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import time, gc

print(f"torch {torch.__version__} | transformers {transformers.__version__}")
print(f"CUDA allocated at import: {torch.cuda.memory_allocated()/1e9:.1f} GB  (must be 0.0 -- if not, RESTART THE KERNEL)")

In [ ]:
# config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

DECODER_MODEL = "google/gemma-4-31b-it"
LEARNING_RATE = 2e-4

# Set these AFTER running the batch-size probe (cell 8). 16 is a safe starting point.
# GRAD_ACCUM only exists to fake a bigger batch when memory is short -- it does NOT
# save time. You have the memory, so keep it at 1.
BATCH_SIZE = 16
GRAD_ACCUM = 1
EVAL_BATCH_SIZE = 32

EPOCHS = 6
PATIENCE = 2
SAVE_ADAPTERS = True

MAX_LEN = 512
N_FOLDS = 5
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
RESULTS_DIR = "results_qlora_gender_gemma31b"
os.makedirs(RESULTS_DIR, exist_ok=True)

SHORT_PROMPT = False       # True = drop the instruction (~4x faster; test on one fold first)
RESTRICT_SOFTMAX = False   # True = CE over the 4 label tokens only, instead of the full 262k vocab

CLASS_LABELS = ["0", "1", "2", "3"]
LABEL_PROXIES = ["Zero", "One", "Two", "Three"]
N_CLASSES = len(CLASS_LABELS)

print(f"Batch {BATCH_SIZE} x accum {GRAD_ACCUM} = effective {BATCH_SIZE*GRAD_ACCUM}")

In [ ]:
# load model

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(DECODER_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    DECODER_MODEL,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    device_map={"": 0},
)

base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base_model.enable_input_require_grads()   # required: without it, autograd won't recompute
                                          # the checkpointed blocks (frozen embeddings)
print("gradient checkpointing: ON")

dmap = getattr(base_model, "hf_device_map", {}) or {}
offloaded = {k: v for k, v in dmap.items() if str(v) in ("cpu", "disk")}
assert not offloaded, f"OFFLOADED LAYERS: {offloaded}"

print(f"Loaded. GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB  (expect ~18-20)")

In [ ]:
# LoRA config + label tokens

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj)",
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)

ANSWER_IDS = []
for name, proxy in zip(CLASS_LABELS, LABEL_PROXIES):
    ids = tokenizer.encode(" " + proxy, add_special_tokens=False)
    assert len(ids) == 1, f"'{proxy}' is {len(ids)} tokens -- pick a single-token proxy"
    ANSWER_IDS.append(ids[0])
    print(f"  {name:15s} -> '{proxy}': {ids}")

ANSWER_IDS_T = torch.tensor(ANSWER_IDS, device=device)

In [ ]:
def _unwrap(m):
    return m.get_base_model() if hasattr(m, "get_base_model") else m

def resolve_trunk(m):
    inner = _unwrap(m)
    for path in ("model.language_model", "language_model", "model.model", "model"):
        obj, ok = inner, True
        for part in path.split("."):
            if not hasattr(obj, part):
                ok = False
                break
            obj = getattr(obj, part)
        if ok and hasattr(obj, "layers") and not hasattr(obj, "lm_head"):
            return obj
    raise RuntimeError("Could not find the text trunk. Top-level modules: "
                       + str([n for n, _ in inner.named_modules() if 0 < n.count(".") <= 1][:20]))

def resolve_head(m):
    head = _unwrap(m).get_output_embeddings()
    assert head is not None, "no lm_head / output embeddings"
    return head

# Gemma has historically applied softcapping to the logits INSIDE ForCausalLM.forward,
# after the lm_head. If this config does, we have to reproduce it or the fast path is
# not equivalent. The assert in the next cell is what actually catches this.
_cfg = base_model.config
LOGIT_SOFTCAP = getattr(_cfg, "final_logit_softcapping", None) or getattr(
    getattr(_cfg, "text_config", None), "final_logit_softcapping", None)

def last_logits(trunk, head, input_ids, attention_mask, position_ids):
    hs = trunk(
        input_ids=input_ids,
        attention_mask=attention_mask,
        position_ids=position_ids,
    ).last_hidden_state[:, -1:, :]          # [B, 1, hidden]  <- only the answer slot
    lg = head(hs)[:, 0, :]                  # [B, vocab]
    if LOGIT_SOFTCAP:
        lg = torch.tanh(lg / LOGIT_SOFTCAP) * LOGIT_SOFTCAP
    return lg

print("trunk:", type(resolve_trunk(base_model)).__name__)
print("head :", type(resolve_head(base_model)).__name__)
print("softcap:", LOGIT_SOFTCAP)

In [ ]:
# load data + pre-tokenize

df = pd.read_excel("../../../data/kostovicova_paskhalis/kostovicova_dqi_text_labels.xlsx")
texts = df["text"].astype(str).tolist()
labels = df["label_main"].astype(int).tolist()   # DQI justification level 0-3
print(f"{len(texts)} speech acts | {df['label_main'].value_counts().sort_index().to_dict()}")

INSTRUCTION = (  ## CHECK IF PROMPT MATCHES CLOSELY ZERO SHOT
    "Code the level of justification of this speech act from a parliamentary debate "
    "(in Serbian, Bosnian, or Croatian; judge the meaning regardless of language) using the "
    "Discourse Quality Index. Answer Zero if the speaker states a demand or position but gives "
    "no reason for it. Answer One if a reason is given but not linked to the demand (inferior "
    "justification). Answer Two if a reason is given and linked to the demand (one complete, "
    "qualified justification). Answer Three if at least two complete justifications are given "
    "(sophisticated justification).\n"
)

def build_prompt(text):
    head = "" if SHORT_PROMPT else INSTRUCTION
    return f'{head}Speech act: "{text}"\nAnswer:'

# Tokenize once, not inside __getitem__ on every epoch of every fold. Some speech acts
# run over MAX_LEN, so truncate the body and re-append the answer suffix -- the prompt
# must end on "Answer:" or the last-slot logic reads the wrong position.
SUFFIX = '"\nAnswer:'
SUFFIX_IDS = tokenizer(SUFFIX, add_special_tokens=False)["input_ids"]

def encode_prompt(text):
    head = "" if SHORT_PROMPT else INSTRUCTION
    body = f'{head}Speech act: "{text}'
    ids = tokenizer(body, add_special_tokens=False, truncation=True,
                    max_length=MAX_LEN - len(SUFFIX_IDS))["input_ids"]
    return ids + SUFFIX_IDS

ALL_IDS = [encode_prompt(t) for t in texts]
_L = np.array([len(x) for x in ALL_IDS])
print(f"prompt tokens: median {int(np.median(_L))}, p95 {int(np.percentile(_L,95))}, max {_L.max()}")


class TaskDataset(Dataset):
    def __init__(self, idxs):
        self.idxs = list(idxs)
    def __len__(self):
        return len(self.idxs)
    def __getitem__(self, i):
        j = self.idxs[i]
        return {"input_ids": ALL_IDS[j], "label": labels[j]}


def collate_fn(batch):
    # LEFT padding: puts the answer slot at index -1 for every row, which is what lets
    # us slice the lm_head to a single position. Verified in the next cell.
    m = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    ids, attn = [], []
    for b in batch:
        k = m - len(b["input_ids"])
        ids.append([pad] * k + b["input_ids"])
        attn.append([0] * k + [1] * len(b["input_ids"]))
    attn = torch.tensor(attn, dtype=torch.long)
    pos = (attn.cumsum(-1) - 1).masked_fill(attn == 0, 1)   # RoPE positions must be mask-aware
    return {
        "input_ids": torch.tensor(ids, dtype=torch.long),
        "attention_mask": attn,
        "position_ids": pos,
        "labels": torch.tensor([b["label"] for b in batch], dtype=torch.long),
    }


def make_loader(idxs, batch_size, shuffle):
    return DataLoader(TaskDataset(idxs), batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collate_fn, pin_memory=True)


@torch.no_grad()
def classify_qlora(idxs, model, batch_size=EVAL_BATCH_SIZE):
    trunk, head = resolve_trunk(model), resolve_head(model)
    was_training = model.training
    model.eval()
    preds = []
    for b in make_loader(idxs, batch_size, shuffle=False):
        lg = last_logits(trunk, head,
                         b["input_ids"].to(device),
                         b["attention_mask"].to(device),
                         b["position_ids"].to(device))
        preds.extend(lg[:, ANSWER_IDS_T].float().argmax(-1).tolist())
    if was_training:
        model.train()
    return preds

In [ ]:
# 5-fold CV

def hms(s):
    s = int(s)
    return f"{s//3600}h{(s%3600)//60:02d}m{s%60:02d}s" if s >= 3600 else f"{s//60}m{s%60:02d}s"

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []
run_t0 = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels)):
    print(f'\n{"="*60}\nFOLD {fold+1}\n{"="*60}')
    print(f'  Train: {len(train_idx)} | Val: {len(val_idx)}')

    model = get_peft_model(base_model, lora_config)

    # get_peft_model() injects into base_model IN PLACE. A leaked adapter would inflate
    # every later fold with no warning. lora_B is zero-init, so nonzero here = leak.
    _lb = [m.weight for n, m in model.named_modules() if n.endswith("lora_B.default")]
    assert _lb and all(torch.count_nonzero(w) == 0 for w in _lb), "adapter leaked from previous fold"
    if fold == 0:
        model.print_trainable_parameters()

    trunk, head = resolve_trunk(model), resolve_head(model)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE)

    train_loader = make_loader(train_idx, BATCH_SIZE, shuffle=True)
    print(f"  Train batches: {len(train_loader)}  <- must be ~{int(np.ceil(len(train_idx)/BATCH_SIZE))}")

    best_f1, best_preds, best_epoch, best_loss = -1.0, None, -1, float("nan")
    patience_counter = 0
    torch.cuda.reset_peak_memory_stats()

    for epoch in range(EPOCHS):
        t0 = time.time()
        print(f'\n  ======== Epoch {epoch+1} / {EPOCHS} ========')
        model.train()
        loss_sum = torch.zeros((), device=device)
        n_seen = n_tokens = 0
        optimizer.zero_grad(set_to_none=True)

        for bi, b in enumerate(train_loader):
            y = b["labels"].to(device, non_blocking=True)
            lg = last_logits(trunk, head,
                             b["input_ids"].to(device, non_blocking=True),
                             b["attention_mask"].to(device, non_blocking=True),
                             b["position_ids"].to(device, non_blocking=True))

            if RESTRICT_SOFTMAX:
                loss = F.cross_entropy(lg[:, ANSWER_IDS_T].float(), y)
            else:
                loss = F.cross_entropy(lg.float(), ANSWER_IDS_T[y])

            (loss / GRAD_ACCUM).backward()
            if (bi + 1) % GRAD_ACCUM == 0 or (bi + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            loss_sum += loss.detach() * y.size(0)
            n_seen += y.size(0)
            n_tokens += int(b["attention_mask"].sum())
            if (bi + 1) % 20 == 0:
                el = time.time() - t0
                eta = el / (bi + 1) * (len(train_loader) - bi - 1)
                print(f'    {bi+1}/{len(train_loader)}  |  {n_tokens/el:,.0f} tok/s  |  '
                      f'{hms(el)} elapsed, {hms(eta)} left    ', end='\r')

        avg_loss = (loss_sum / n_seen).item()
        et = time.time() - t0
        print(f'\n    Loss {avg_loss:.4f}  |  {hms(et)}  |  {n_tokens/et:,.0f} tok/s  '
              f'|  peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB')

        tv = time.time()
        val_preds = classify_qlora(val_idx, model)
        y_val = [labels[i] for i in val_idx]
        f1m = f1_score(y_val, val_preds, average="macro", zero_division=0)
        f1w = f1_score(y_val, val_preds, average="weighted", zero_division=0)
        print(f'    Val Macro F1 {f1m:.4f} | Weighted {f1w:.4f} | {hms(time.time()-tv)}')

        if f1m > best_f1:
            best_f1, best_preds, best_epoch, best_loss = f1m, val_preds, epoch + 1, avg_loss
            patience_counter = 0
            if SAVE_ADAPTERS:
                model.save_pretrained(f"{RESULTS_DIR}/fold{fold+1}_best_adapter")
            print(f'    ** best ({best_f1:.4f}) **')
        else:
            patience_counter += 1
            print(f'    no improvement ({patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'    early stop at epoch {epoch+1}')
            break

    y_true = [labels[i] for i in val_idx]
    f1_macro = f1_score(y_true, best_preds, average="macro", zero_division=0)
    f1_weighted = f1_score(y_true, best_preds, average="weighted", zero_division=0)
    per_class = f1_score(y_true, best_preds, average=None, labels=list(range(N_CLASSES)), zero_division=0)

    fm = {"fold": fold + 1, "best_epoch": best_epoch, "train_loss": round(best_loss, 4),
          "f1_macro": round(f1_macro, 4), "f1_weighted": round(f1_weighted, 4)}
    for i, lab in enumerate(CLASS_LABELS):
        fm[f"f1_{lab}"] = round(float(per_class[i]), 4)
    fold_results.append(fm)

    print(f"\n  Best epoch {best_epoch} | Macro F1 {f1_macro:.4f} | Weighted {f1_weighted:.4f}")
    for i, lab in enumerate(CLASS_LABELS):
        print(f"    {lab:15s}: {per_class[i]:.3f}")

    pd.DataFrame(fold_results).to_csv(f"{RESULTS_DIR}/cv_results_running.csv", index=False)

    # projected total, based on folds finished so far
    elapsed = time.time() - run_t0
    projected = elapsed / (fold + 1) * N_FOLDS
    print(f"  Saved. Elapsed {hms(elapsed)} | projected total {hms(projected)} "
          f"| {hms(projected - elapsed)} remaining")

    try:
        base_model = model.unload()
    except Exception as e:
        print(f"  (unload unavailable: {e} -- fold-start assert is the safety net)")
    del model, optimizer, trainable, trunk, head
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n{"="*60}\nDONE in {hms(time.time()-run_t0)}\n{"="*60}')

In [ ]:
# summary

results_df = pd.DataFrame(fold_results)
results_df.to_csv(f"{RESULTS_DIR}/cv_results_final.csv", index=False)

print("=== QLoRA Gemma-4 31B: KOSTOVICOVA-PASKHALIS DQI JUSTIFICATION CV RESULTS ===")
print(f"Macro F1:    {results_df['f1_macro'].mean():.4f} (SD {results_df['f1_macro'].std():.4f})")
print(f"Weighted F1: {results_df['f1_weighted'].mean():.4f} (SD {results_df['f1_weighted'].std():.4f})")
print("\nPer-class F1 (mean across folds):")
for lab in CLASS_LABELS:
    print(f"  {lab:15s}: {results_df[f'f1_{lab}'].mean():.4f} (SD {results_df[f'f1_{lab}'].std():.4f})")
print(f"\nBest epoch per fold: {results_df['best_epoch'].tolist()}")
print()
print(results_df.to_string(index=False))